In [ ]:
import duckdb
import pandas as pd
import os

In [ ]:
con = duckdb.connect()
con.execute("SET threads = 2")
con.execute("SET memory_limit = '2GB'")
temp_dir = os.path.abspath("../data/interim/duckdb_tmp")
os.makedirs(temp_dir, exist_ok=True)
con.execute("SET temp_directory = ?", [temp_dir])

# Check if raw data exists
raw_path = "../data/raw/food.parquet"
if not os.path.exists(raw_path):
    raise FileNotFoundError(f"Raw data not found at {raw_path}")

query = """
WITH filtered_products AS (
    SELECT
        f.code,
        f.brands,
        list_extract(list_transform(list_filter(f.product_name, p -> p.lang = 'en'), p -> p.text), 1) AS product_name_en,
        list_extract(list_transform(list_filter(f.ingredients_text, i -> i.lang = 'en'), i -> i.text), 1) AS ingredients_text_en,
        f.ingredients,
        f.ingredients_tags,
        f.allergens_tags,
        f.traces_tags,
        f.categories,
        f.countries_tags
    FROM '../data/raw/food.parquet' AS f
    WHERE
        list_contains(f.countries_tags, 'en:philippines')
        AND f.ingredients IS NOT NULL
)
SELECT DISTINCT
    code,
    brands,
    product_name_en,
    ingredients_text_en,
    ingredients,
    ingredients_tags,
    allergens_tags,
    traces_tags,
    categories,
    countries_tags
FROM filtered_products
WHERE ingredients_text_en IS NOT NULL
"""

In [ ]:
try:
    df = con.execute(query).fetchdf()
    print(f"Extracted {len(df)} rows")
except Exception as e:
    print(f"Query failed: {e}")
    raise

# Check empty ingredient texts (should be none)
empty_ingredients = (df['ingredients_text_en'].str.len() == 0).sum()
print(f"Rows with empty ingredients_text_en: {empty_ingredients}")

# Drop duplicates (already distinct, but safe)
df = df.drop_duplicates(subset=["code"])
print(f"After dropping duplicates: {len(df)} rows")

In [ ]:
# Ensure the interim directory exists
os.makedirs("../data/interim/", exist_ok=True)

# Save to interim
df.to_csv("../data/interim/extracted_raw.csv", index=False)
print("Saved to ../data/interim/extracted_raw.csv")